<a href="https://colab.research.google.com/github/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/blob/rag/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [24]:
PARQUET_PATH = '/content/drive/MyDrive/Progetto-NLP/Branch-rag/collection_ita.parquet'
EMBEDDINGS_PATH = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita"
VECTOR_DB_PATH = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita/db"

'''
Progetto-NLP/
├─ Branch-rag/
│  ├─ embeddings_collection_ita/
│  │  ├─ status_registry.json   #mancanti vs completati
│  │  ├─ embeddings_chunk_i_i+10k.pkl
│  │  ├─ embeddings_...
│  │  ├─ db/
│  │  │  ├─ db_registry.json   #elementi aggiunti al db

'''

'\nProgetto-NLP/\n├─ Branch-rag/\n│  ├─ embeddings_collection_ita/\n│  │  ├─ status_registry.json   #mancanti vs completati\n│  │  ├─ embeddings_chunk_i_i+10k.pkl\n│  │  ├─ embeddings_...\n│  │  ├─ db/\n│  │  │  ├─ db_registry.json   #elementi aggiunti al db\n\n'

In [9]:
# import phase
!pip install beir rank_bm25 faiss-cpu ir_measures tqmd chromadb #hnswlib

from rank_bm25 import BM25Okapi
import numpy as np
import faiss
#import hnswlib
import time
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import torch
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from datasets import load_dataset
import chromadb
import tqmd

import os


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 11.3 MB/s eta 0:00:00


## Load `.parquet` file

In [ ]:
if (os.path.exists(PARQUET_PATH)):
  print("Ok")

ds = load_dataset("parquet", data_files=PARQUET_PATH, split="train")

# Get total number of documents without loading them
num_docs = len(ds)
corpus_ids = list(range(num_docs))

print(f"Total documents: {num_docs}")

Ok


Generating train split: 0 examples [00:00, ? examples/s]

# Management code for `status_registry.json`

In [5]:
#This code is useful to update status_registry.json: it moves files names from "mancanti" to "completati" based on their presence or not in the output folder
import os

from pathlib import Path
import json

num_docs=8211278
totale_count=0
chunk_size=10000

import re


def sort_key(filename):
    # extracts all numbers from filename as integers
    return [int(x) for x in re.findall(r"\d+", filename)]


def init_registry(num_docs, chunk_size, embedding_path):
    registry_file = os.path.join(embedding_path, "status_registry.json")
    if os.path.exists(registry_file):
        with open(registry_file, 'r') as f:
            registry = json.load(f)
    else:
        print("[Error] Register not found!")

    tutti_i_file_attesi = [
      f"embeddings_chunk_{i}_{min(i + chunk_size, num_docs)}.pkl"
      for i in range(0, num_docs, chunk_size)]

    for filename in tutti_i_file_attesi:
          # 2. Aggiorna le liste
          if filename not in registry["mancanti"]:
              print(filename)
              registry["mancanti"].append(filename)
          with open(registry_file, 'w') as f:
            json.dump(registry, f, indent=4)


#makes all fields from the registry empty
def empty_registry_fields(embedding_path):
    registry_file = os.path.join(embedding_path, "status_registry.json")
    if os.path.exists(registry_file):
        with open(registry_file, 'r') as f:
            registry = json.load(f)

    registry["mancanti"] = []
    registry["completati"] = []


    with open(registry_file, 'w') as f:
        json.dump(registry, f, indent=4)

# Checks files contained in the folder, removes them from "mancanti" and puts it in "completati", recomputes progress percentage
def update_registry_from_directory(embedding_path):

    #compute total number of chunks
    totale_count=num_docs/chunk_size

    #retrieves chunks filename already in the directory
    files=[]
    folder_path = Path(embedding_path)
    for file_path in folder_path.iterdir():
        if file_path.is_file():
          if file_path.name!="status_registry.json":
            files.append(file_path.name)

    #Legge il registro esistente
    registry_file = os.path.join(embedding_path, "status_registry.json")
    if os.path.exists(registry_file):
        with open(registry_file, 'r') as f:
            registry = json.load(f)
    else:
        print("[Error] Register not found!")

    #sorts based on key (chunk number in the filename)
    sorted_files = sorted(files, key=sort_key)

    #moves from "mancanti" to "completati"
    for filename in sorted_files:
        #print(filename)
        # 2. Aggiorna le liste
        if filename not in registry["completati"]:
            registry["completati"].append(filename)
        if filename in registry["mancanti"]:
            registry["mancanti"].remove(filename)

    # Calcola la percentuale di completamento totale
    completati_count = len(registry["completati"])
    registry["percentuale_completamento"] = f"{(completati_count / totale_count) * 100:.2f}%"

    # 3. Salva il registro aggiornato su Drive
    with open(registry_file, 'w') as f:
        json.dump(registry, f, indent=4)

def get_missing_ranges(embedding_path):

    registry_file = os.path.join(embedding_path, "status_registry.json")

    with open(registry_file, "r") as f:
        registry = json.load(f)

    mancanti = registry["mancanti"]

    ranges = []

    # Estrae start/end dai filename
    for filename in mancanti:

        match = re.search(
            r"embeddings_chunk_(\d+)_(\d+)\.pkl",
            filename
        )

        if match:
            start = int(match.group(1))
            end = int(match.group(2))
            ranges.append((start, end))

    # Ordina per start
    ranges.sort(key=lambda x: x[0])

    # Merge intervalli consecutivi
    merged = []

    for current_start, current_end in ranges:

        if not merged:
            merged.append([current_start, current_end])

        else:
            last_start, last_end = merged[-1]

            # Se consecutivi o sovrapposti
            if current_start <= last_end:
                merged[-1][1] = max(last_end, current_end)

            else:
                merged.append([current_start, current_end])

    tot_chunks = 0
    for start, end in merged:
      tot_chunks += (end-start)/10000

    print(f"tot chunks: {tot_chunks}")

    return merged

#init_registry()
#empty_registry_completati_field()
update_registry_from_directory(EMBEDDINGS_PATH)
print(get_missing_ranges(EMBEDDINGS_PATH))







tot chunks: 206.1278
[[920000, 1000000], [1810000, 2000000], [3110000, 4000000], [4090000, 4120000], [4480000, 4500000], [5850000, 6500000], [8010000, 8211278]]


In [ ]:

bi_enc = SentenceTransformer('BAAI/bge-m3', model_kwargs={"torch_dtype": torch.float16},)
#bi_enc.max_seq_length = 1024
#x_enc = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def embed_sparse(docs):
    return list(map(lambda text: text.split(" "), docs))

def embed_dense(encoder, docs, batch_size):
    return encoder.encode(docs, convert_to_numpy=True, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)

def dense_search(query, top_k):
    # get similarity scores
    query_embedding = embed_dense(bi_enc, [query])[:1]
    doc_inds, scores = index_hnsw.knn_query(query_embedding, k=top_k)

    # format to run
    run = {corpus_ids[int(ind)]: float(s) for ind, s in zip(doc_inds[0], scores[0])}
    return run

In [ ]:
# create dense index processing dataset in chunks and storing them in the drive at the specified path
bi_enc.max_seq_length = 1024
import os
import pickle
from concurrent.futures import ThreadPoolExecutor

# Assicura che la cartella esista
os.makedirs(EMBEDDINGS_PATH, exist_ok=True)

chunk_size = 10000  # Numero di documenti per ogni salvataggio
start_index = 3050000    # Cambia questo valore se devi ripartire da un punto interrotto
stop_index= 4000000
print(stop_index)

internal_batch_size = 4

def save_to_disk(data, filename):
    with open(os.path.join(EMBEDDINGS_PATH, filename), 'wb') as f:
        pickle.dump(data, f)
    print(f"Completato salvataggio: {filename}")

# Sostituiamo len(corpus_docs) con num_docs (che abbiamo definito prima)
with ThreadPoolExecutor(max_workers=2) as executor:
  for i in range(start_index, num_docs, chunk_size):
      end_index = min(i + chunk_size, num_docs)
      if end_index >= stop_index:
        print("Finished embedding the requested slide")
        break

      print(f"Processando blocchi da {i} a {end_index}...")

      # CHANGED HERE: Estraiamo la fetta direttamente dal dataset su disco.
      # Questo carica in RAM SOLO i 10000 documenti di questo blocco.
      batch_docs = ds[i:end_index]['content']

      # Genera gli embedding (usa la tua funzione embed_dense con batch_size=4)
      embeddings = embed_dense(bi_enc, batch_docs, batch_size=internal_batch_size)

      # Salva il pezzetto su Drive
      file_name = f"embeddings_chunk_{i}_{end_index}.pkl"
      executor.submit(save_to_disk, embeddings, file_name)


      # 5. PULIZIA BRUTALE DELLA MEMORIA
      del batch_docs
      del embeddings
      torch.cuda.empty_cache()      # Svuota la VRAM della GPU

print("Tutti i processi di calcolo sono terminati. I salvataggi finali potrebbero richiedere ancora qualche secondo.")

## Create Index

In [ ]:
# import os
# import pickle
# import numpy as np
# import re
# import hnswlib
# import gc

# path = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embeddings_subset_it"

# # 1. Get and sort files
# if not os.path.exists(path):
#   os.makedirs(path)
# files = [f for f in os.listdir(path) if f.endswith('.pkl')]
# def extract_number(filename):
#     match = re.search(r'chunk_(\d+)_', filename)
#     return int(match.group(1)) if match else 0
# files.sort(key=extract_number)

# # 2. Initialize HNSW Index FIRST
# dim = 1024 # BAAI/bge-m3 dimension
# index_hnsw = hnswlib.Index(space='cosine', dim=dim)
# index_hnsw.init_index(max_elements=num_docs, ef_construction=200, M=32)

# # 3. Incrementally load, add to index, and delete from RAM
# current_id = 0
# for file in files:
#     with open(os.path.join(path, file), 'rb') as f:
#         chunk_embeddings = pickle.load(f)

#     chunk_size = chunk_embeddings.shape[0]
#     ids_for_chunk = list(range(current_id, current_id + chunk_size))

#     # Add to index
#     index_hnsw.add_items(chunk_embeddings, ids_for_chunk)
#     print(f"Aggiunto all'indice: {file} (Elementi: {chunk_size})")

#     current_id += chunk_size

#     # FREE MEMORY explicitly
#     del chunk_embeddings
#     del ids_for_chunk
#     gc.collect()

# print("Indice popolato con successo!")

# # Initialize the generative component
# from transformers import pipeline, AutoTokenizer
# pipe = pipeline("text-generation", model="meta-llama/Llama-3.2-1B")
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

In [ ]:
# #Save complete embeddings at the specified path
# index_path = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embeddings/kurdish_wiki_index.bin"
# index_hnsw.save_index(index_path)
# print("Indice salvato! Ora puoi respirare.")

In [14]:
!pip install tqdm

In [25]:
import os
import pickle
import re
import chromadb
import gc
import tqdm
from concurrent.futures import ThreadPoolExecutor

MAX_BATCH_SIZE = 5000

def update_db_registry(db_registry, file):
    db_registry.add(file)
    with open(db_registry_file, 'w') as registry_f:
        json.dump(list(db_registry), registry_f)



# Cartella dove ChromaDB salverà il DATABASE VERO E PROPRIO
os.makedirs(VECTOR_DB_PATH, exist_ok=True)

# 1. Inizializza il Client Persistente (Il vero "servizio" di Database)
client = chromadb.PersistentClient(path=VECTOR_DB_PATH)

# Crea o recupera la "Collection" (la tabella del database)
# Usiamo la distanza 'cosine' per combaciare con BAAI/bge-m3
collection = client.get_or_create_collection(
    name="wiki_rag_collection",
    metadata={"hnsw:space": "cosine"}
)

# Create or open db_registry.json
db_registry_file = os.path.join(VECTOR_DB_PATH, "db_registry.json")

if os.path.exists(db_registry_file):
    with open(db_registry_file, 'r') as registry_f:
            db_registry = set(json.load(registry_f))
else:
    db_registry = set()

# Take embeddings .plk files from directory and sorts for chunk number
files = [f for f in os.listdir(EMBEDDINGS_PATH) if f.endswith('.pkl')]

def extract_number(filename):
    match = re.search(r'chunk_(\d+)_', filename)
    return int(match.group(1)) if match else 0

files.sort(key=extract_number)

# 3. Inserimento Incrementale nel Database
current_id = 0

with ThreadPoolExecutor(max_workers=2) as executor:
  with tqdm.tqdm(total=(len(files)-len(db_registry)), desc="Popolamento Database", unit="file") as pbar:
    for file in files:

      if file in db_registry:
        #print(f"File già inserito nel database: {file}")
        continue

      current_id = extract_number(file)

      with open(os.path.join(EMBEDDINGS_PATH, file), 'rb') as embedding_f:

          chunk_embeddings = pickle.load(embedding_f)

      chunk_size = chunk_embeddings.shape[0]

      # ChromaDB richiede che gli ID siano stringhe (es. "id_0", "id_1")
      ids_for_chunk = [str(i) for i in range(current_id, current_id + chunk_size)]

      for i in range(0, chunk_size, MAX_BATCH_SIZE):
          # Calcoliamo la fine del sotto-batch
          end_idx = min(i + MAX_BATCH_SIZE, chunk_size)

          # Estraiamo la fetta (slice) di embedding e ID
          sub_batch_embeddings = chunk_embeddings[i:end_idx].tolist()
          sub_batch_ids = ids_for_chunk[i:end_idx]

          # Inserimento del sotto-batch
          collection.add(
              embeddings=sub_batch_embeddings,
              ids=sub_batch_ids
          )

      pbar.set_postfix({"ultimo_file": file})
      pbar.update(1)


      executor.submit(update_db_registry, db_registry, file)



      current_id += chunk_size

      # Libera la RAM
      del chunk_embeddings
      del ids_for_chunk
      gc.collect()

print("Database Chroma popolato con successo su Google Drive!")

Popolamento Database:   0%|          | 0/618 [00:03<?, ?file/s]


KeyboardInterrupt: 

In [ ]:
# Initialize the generative component
from transformers import pipeline, AutoTokenizer
from huggingface_hub import login

from google.colab import userdata
TOKEN=userdata.get('HF_TOKEN')

import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

pipe = pipeline("text-generation", model="meta-llama/Llama-3.2-1B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")


In [ ]:

def rag(query, top_k):
    # create system and user messages
    system_prompt = "You are helpful assistant. Answer the question given the supportive information."
    user_prompt = f"Question: {query}"

    # search relevant documents
    if top_k > 0:
        # runs = rrf_search(query, top_k)
        runs = dense_search(query, top_k)
        docs = [ds[int(doc_id)]['content'] for doc_id in runs]
        docs_context = "\n".join(docs)
        user_prompt += f"\n Referenced knowledge: {docs_context}"

    # format prompt
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # generate the answer
    outputs = pipe(
        prompt,
        max_new_tokens=128,
        do_sample=True,
        max_length=None,
        temperature=1.0,
        eos_token_id=pipe.tokenizer.eos_token_id,
        pad_token_id=pipe.tokenizer.pad_token_id
    )
    predicted_answer = outputs[0]['generated_text'][len(prompt):].strip()

    if top_k > 0:
      return prompt, predicted_answer, docs
    else: return prompt, predicted_answer

In [ ]:
  top_k = 0
  prompt, predicted_answer = rag("Chi ha vinto il AEGON Pro Series Edgbaston 2013?", top_k)
  print(f"Prompt: {prompt}\nPredicted answer: {predicted_answer}")

In [ ]:
top_k = 2
prompt, predicted_answer, docs = rag("who was the winner of AEGON Pro Series Edgbaston 2013?", top_k)
i=0
for doc in docs:
  print(f"===============================\nDoc {i}\n:{doc}\n===============================")
  i=i+1
print("==================Answer=============")
print(f"Prompt: {prompt}\nPredicted answer: {predicted_answer}")

In [ ]:
#Debug: test single query vs document
# 1. Prendi il blocco "incriminato" e la tua domanda
target_doc = """<Table>
| 10 cm M>  8 Gebirgshaubitze |
| --- | --- |
M. 10
Tipo: obice da montagna
Origine: Austria-Ungheria
Impiego
Utilizzatori: KuK Armee
Conflitti: prima guerra mondiale
Produzione
Data progettazione: 1908-1910
Date di produzione: 1908-1916
Ritiro dal servizio: 1918
Varianti: 10 cm M. 10 Gebirgshaubitze
Descrizione
Calibro: 104 mm
Peso proiettile: 14,3 kg
Azionamento: otturatore a vite interrotta eccentrico
Elevazione: +43°
voci di armi d'artiglieria presenti su Wikipedia
</Table>
Il 10 cm Gebirgshaubitze M. 8 era un obice da montagna austro-ungarico, impiegato durante la prima guerra mondiale.
"""
query = "Qual è il peso del proiettile 10 cm Gebirgshaubitze M. 8?"

# 2. Genera gli embedding solo per questi due
target_emb = bi_enc.encode(["passage: " + target_doc], normalize_embeddings=True)
query_emb = bi_enc.encode([query], normalize_embeddings=True)

# 3. Calcola la similarità coseno manuale (deve essere alta, es. > 0.7)
similarity = np.dot(query_emb, target_emb.T)
print(f"Similarità tra domanda e blocco specifico: {similarity[0][0]:.4f}")